# Distance vs SNR Analysis

Explore how signal strength varies with path distance.

**What you'll learn:**
- The relationship between distance and SNR
- Skip zone effects on different bands
- Optimal distance ranges for each band

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import load_dataset, plot_distance_snr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
df = load_dataset("wspr")
print(f"Loaded {len(df):,} signatures")
print(f"Distance range: {df['avg_distance'].min():.0f} - {df['avg_distance'].max():.0f} km")

## Distance and Ionospheric Propagation

HF signals reflect off the ionosphere. The distance they travel depends on:

- **Takeoff angle:** Lower angles = longer skip distance
- **Layer height:** F2 layer (~300 km) allows longer hops than E layer (~100 km)
- **MUF:** Maximum Usable Frequency limits which bands can reach which distances

**Skip zone:** The area between ground wave range (~100 km) and first ionospheric hop where no signal is received.

In [ ]:
# Distance distribution
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df["avg_distance"], bins=100, alpha=0.7, edgecolor="black")
ax.axvline(500, color="red", linestyle="--", label="Ground wave cutoff (500 km)")
ax.axvline(3000, color="orange", linestyle="--", label="Single hop typical (3000 km)")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Path Distances")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# SNR vs Distance for all bands
plot_distance_snr(df, bins=30)
plt.show()

In [ ]:
# Per-band analysis
BANDS = [103, 105, 107, 109, 111]  # 80m, 40m, 20m, 15m, 10m
BAND_NAMES = {103: "80m", 105: "40m", 107: "20m", 109: "15m", 111: "10m"}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, band in enumerate(BANDS):
    band_df = df[df["band"] == band].copy()
    
    if len(band_df) > 100:
        # Bin distances
        band_df["dist_bin"] = pd.cut(band_df["avg_distance"], bins=20)
        agg = band_df.groupby("dist_bin")["median_snr"].agg(["mean", "std", "count"]).reset_index()
        agg["dist_center"] = agg["dist_bin"].apply(lambda x: x.mid if pd.notna(x) else np.nan)
        agg = agg.dropna()
        
        axes[i].errorbar(agg["dist_center"], agg["mean"], 
                        yerr=agg["std"]/np.sqrt(agg["count"]),
                        fmt="o-", capsize=3)
        axes[i].axhline(0, color="gray", linestyle="--", alpha=0.5)
        axes[i].set_xlabel("Distance (km)")
        axes[i].set_ylabel("Mean SNR (dB)")
        axes[i].set_title(f"{BAND_NAMES[band]} Band")
        axes[i].grid(True, alpha=0.3)

# Hide empty subplot
axes[-1].axis("off")

plt.suptitle("SNR vs Distance by Band", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay comparison
fig, ax = plt.subplots(figsize=(14, 7))

for band in BANDS:
    band_df = df[df["band"] == band].copy()
    
    if len(band_df) > 100:
        band_df["dist_bin"] = (band_df["avg_distance"] // 500) * 500
        agg = band_df.groupby("dist_bin")["median_snr"].mean()
        ax.plot(agg.index, agg.values, marker="o", 
                label=BAND_NAMES[band], linewidth=2, markersize=4)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Mean SNR (dB)")
ax.set_title("SNR vs Distance — All Bands Compared")
ax.legend(title="Band")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Observations

**Low bands (80m, 160m):**
- Better SNR at shorter distances (< 2000 km)
- Multiple-hop paths (> 5000 km) have significant loss
- Ground wave effective to ~500 km

**High bands (10m, 15m):**
- Skip zone visible (poor SNR at short distances)
- Sweet spot around 2000-5000 km (single F2 hop)
- Long paths (> 10000 km) viable during solar max

**Mid bands (20m, 40m):**
- Most versatile — work at most distances
- 20m: good for DX at all distances
- 40m: excellent for regional (500-2000 km)